# Notebook 01 — Collect Raw FRED / ALFRED Data

## Goal
Download raw macroeconomic time series from FRED (current data) and ALFRED (real-time vintage data).
Save them **untouched** to Google Drive. No cleaning happens here.

**Series collected:** PAYEMS, UNRATE, ICSA, JTSJOL, CES0500000003, CPIAUCSL, FEDFUNDS, INDPRO

---

## What can go wrong
- `FRED_API_KEY` not set → all API calls will fail with 403
- ICSA is weekly; it needs aggregation to monthly in a later notebook
- JTSJOL has a ~2-month publication lag; check `max_observation_date`
- ALFRED vintage download is slow (many calls); be patient
- Series discontinued or renamed by FRED → check series IDs
- Rate limiting → adds sleep between calls automatically

---

## What you must inspect
- Are all 8 series present?
- Are date ranges correct? (should start ~2000, end near today)
- Are all values numeric (not None/NaN)?
- Are vintage fields (realtime_start, realtime_end) present in ALFRED data?
- Are there any unexpected missing months?

In [ ]:
import os
import time
import json
import pathlib
import datetime

import pandas as pd
import numpy as np
import requests
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

RAW_FRED_DIR = DRIVE_ROOT / 'data' / 'raw' / 'fred'
RAW_ALFRED_DIR = DRIVE_ROOT / 'data' / 'raw' / 'alfred'
DQ_DIR = DRIVE_ROOT / 'outputs' / 'data_quality'
APPROVALS_DIR = DRIVE_ROOT / 'approvals'

FRED_API_KEY = os.getenv('FRED_API_KEY')
if not FRED_API_KEY:
    raise EnvironmentError('STOP: FRED_API_KEY is required for FRED/ALFRED. Set it in Notebook 00.')

with open(REPO_DIR / 'configs' / 'fred_series.yaml') as f:
    fred_cfg = yaml.safe_load(f)

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)

SERIES_IDS = [s['id'] for s in fred_cfg['series']]
_s = base_cfg['test_sample'] if base_cfg.get('test_mode', False) else base_cfg['sample']
START_DATE = _s['start_date']
END_DATE = _s['end_date']

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode       : {MODE}')
print(f'Window     : {START_DATE} to {END_DATE}')
print(f'Series     : {SERIES_IDS}')
print(f'FRED key   : {FRED_API_KEY[:6]}...')
print()
print('To switch modes, set test_mode: true/false in configs/base.yaml')

## Part A — Download FRED current data

In [ ]:
def fetch_fred_series(series_id, api_key, observation_start='1990-01-01', observation_end='2025-12-31'):
    """Download one FRED series. Returns a DataFrame with columns [series_id, observation_date, value]."""
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'observation_start': observation_start,
        'observation_end': observation_end,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    obs = data.get('observations', [])
    if not obs:
        return pd.DataFrame(columns=['series_id', 'observation_date', 'value'])
    df = pd.DataFrame(obs)[['date', 'value']].rename(columns={'date': 'observation_date'})
    df['series_id'] = series_id
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['observation_date'] = pd.to_datetime(df['observation_date'])
    return df[['series_id', 'observation_date', 'value']]

In [ ]:
from tqdm import tqdm

fred_frames = []
for sid in tqdm(SERIES_IDS, desc='Downloading FRED series'):
    df = fetch_fred_series(sid, FRED_API_KEY, START_DATE, END_DATE)
    fred_frames.append(df)
    time.sleep(0.5)

fred_df = pd.concat(fred_frames, ignore_index=True)
print(f'FRED current data: {fred_df.shape[0]} rows, {fred_df["series_id"].nunique()} series')
print(fred_df.head())

In [ ]:
print('=== FRED Current Data Summary ===')
summary_rows = []
for sid in SERIES_IDS:
    sub = fred_df[fred_df['series_id'] == sid]
    summary_rows.append({
        'series_id': sid,
        'row_count': len(sub),
        'min_observation_date': sub['observation_date'].min(),
        'max_observation_date': sub['observation_date'].max(),
        'missing_values': sub['value'].isna().sum(),
        'min_value': sub['value'].min(),
        'max_value': sub['value'].max(),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

missing_series = [sid for sid in SERIES_IDS if sid not in fred_df['series_id'].values]
if missing_series:
    print(f'\nWARNING: Missing series: {missing_series}')

In [ ]:
fred_path = RAW_FRED_DIR / 'fred_current.parquet'
fred_df.to_parquet(fred_path, index=False)
print(f'Saved FRED current data: {fred_path}')
print(f'File size: {fred_path.stat().st_size / 1024:.1f} KB')

## Part B — Download ALFRED vintage data

ALFRED returns all historical *vintages* of a series — every time the value was revised, a new record is created. This lets us use the value that was *available at the forecast date*, not the final revised value.

**`realtime_start`**: the date from which this observation value was the official release  
**`realtime_end`**: the date until which this value was the official release  

This download can take several minutes due to the volume of vintage records.

In [ ]:
def fetch_alfred_vintage(series_id, api_key, observation_start='1990-01-01', observation_end='2025-12-31'):
    """Download all vintage observations for one ALFRED series."""
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'observation_start': observation_start,
        'observation_end': observation_end,
        'realtime_start': '1970-01-01',
        'realtime_end': '9999-12-31',
        'output_type': '4',  # all vintages
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    obs = data.get('observations', [])
    if not obs:
        return pd.DataFrame()
    df = pd.DataFrame(obs)
    df['series_id'] = series_id
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    for col in ['date', 'realtime_start', 'realtime_end']:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    df = df.rename(columns={'date': 'observation_date'})
    return df[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'value']]

In [ ]:
alfred_frames = []
for sid in tqdm(SERIES_IDS, desc='Downloading ALFRED vintage'):
    try:
        df = fetch_alfred_vintage(sid, FRED_API_KEY, START_DATE, END_DATE)
        alfred_frames.append(df)
        print(f'  {sid}: {len(df)} vintage observations')
    except Exception as e:
        print(f'  {sid}: ERROR — {e}')
    time.sleep(1)

alfred_df = pd.concat(alfred_frames, ignore_index=True)
print(f'\nALFRED vintage data: {alfred_df.shape[0]} rows, {alfred_df["series_id"].nunique()} series')

In [ ]:
print('=== ALFRED Vintage Data Summary ===')
alfred_summary_rows = []
for sid in SERIES_IDS:
    sub = alfred_df[alfred_df['series_id'] == sid]
    alfred_summary_rows.append({
        'series_id': sid,
        'row_count': len(sub),
        'min_observation_date': sub['observation_date'].min(),
        'max_observation_date': sub['observation_date'].max(),
        'min_realtime_start': sub['realtime_start'].min(),
        'max_realtime_start': sub['realtime_start'].max(),
        'missing_values': sub['value'].isna().sum(),
        'vintage_fields_present': all(c in sub.columns for c in ['realtime_start', 'realtime_end']),
    })

alfred_summary = pd.DataFrame(alfred_summary_rows)
print(alfred_summary.to_string(index=False))

In [ ]:
alfred_path = RAW_ALFRED_DIR / 'alfred_vintage.parquet'
alfred_df.to_parquet(alfred_path, index=False)
print(f'Saved ALFRED vintage data: {alfred_path}')
print(f'File size: {alfred_path.stat().st_size / 1024:.1f} KB')

## Save data quality report

In [ ]:
combined_report = pd.merge(summary_df, alfred_summary[['series_id', 'row_count', 'vintage_fields_present']],
                           on='series_id', suffixes=('_fred', '_alfred'))
dq_path = DQ_DIR / 'fred_alfred_collection_report.csv'
combined_report.to_csv(dq_path, index=False)
print(f'Data quality report saved: {dq_path}')

## Assertion checks

In [ ]:
print('Running assertion checks...')

# All series present in FRED
fred_series_found = set(fred_df['series_id'].unique())
missing_from_fred = set(SERIES_IDS) - fred_series_found
assert not missing_from_fred, f'MISSING FRED SERIES: {missing_from_fred}'
print('  All series present in FRED current data.')

# All series present in ALFRED
alfred_series_found = set(alfred_df['series_id'].unique())
missing_from_alfred = set(SERIES_IDS) - alfred_series_found
assert not missing_from_alfred, f'MISSING ALFRED SERIES: {missing_from_alfred}'
print('  All series present in ALFRED vintage data.')

# ALFRED has vintage fields
for col in ['realtime_start', 'realtime_end']:
    assert col in alfred_df.columns, f'Missing ALFRED column: {col}'
print('  ALFRED vintage fields (realtime_start, realtime_end) present.')

# PAYEMS is present (it's the target)
payems_fred = fred_df[fred_df['series_id'] == 'PAYEMS']
assert len(payems_fred) > 100, 'PAYEMS has too few observations'
assert payems_fred['value'].isna().sum() == 0 or payems_fred['value'].isna().mean() < 0.02, \
    'PAYEMS has too many missing values'
print('  PAYEMS target series has sufficient data.')

# No entirely null series
for sid in SERIES_IDS:
    sub = fred_df[fred_df['series_id'] == sid]
    null_pct = sub['value'].isna().mean()
    if null_pct > 0.05:
        print(f'  WARNING: {sid} has {null_pct:.1%} missing values')

print('\nAll assertion checks passed.')

## Manual Approval Gate

**Before approving, please verify:**
1. All 8 series are present in both FRED and ALFRED tables above
2. Date ranges look correct (≥ 2000, ends near today)
3. Values are numeric (no suspicious NaN patterns)
4. ALFRED vintage fields (`realtime_start`, `realtime_end`) are populated
5. No unexpected missing months in PAYEMS

If everything looks correct, change `APPROVE_THIS_STEP = True` and rerun this cell.

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
import datetime
import json

approval = {
    'approved': True,
    'notebook': '01_collect_raw_fred_alfred.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [],
    'output_artifacts': [
        str(fred_path),
        str(alfred_path),
        str(dq_path),
    ],
    'row_counts': {
        'fred_current': int(len(fred_df)),
        'alfred_vintage': int(len(alfred_df)),
    },
    'warnings': [],
    'notes': ''
}

approval_path = APPROVALS_DIR / '01_raw_fred_alfred_approved.json'
with open(approval_path, 'w') as f:
    json.dump(approval, f, indent=2)

print(f'Approval saved: {approval_path}')
print('Notebook 01 complete. Proceed to 02_collect_raw_gdelt.ipynb')